In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

In [ ]:
data = pd.read_csv('criteo_sampled_data.csv')

In [ ]:
data.head(5)

In [ ]:
cols = data.columns.values


# ?????

In [ ]:
# ?????
dense_feats = [f for f in cols if f[0] == "I"]
sparse_feats = [f for f in cols if f[0] == "C"]

# ??dense??
def process_dense_feats(data, feats):
    d = data.copy()
    d = d[feats].fillna(0.0)
    for f in feats:
        d[f] = d[f].apply(lambda x: np.log(x+1) if x > -1 else -1)
    
    return d
data_dense = process_dense_feats(data, dense_feats)

# ??sparse??
def process_sparse_feats(data, feats):
    d = data.copy()
    d = d[feats].fillna("-1")
    for f in feats:
        label_encoder = LabelEncoder()
        d[f] = label_encoder.fit_transform(d[f])
        
    return d
data_sparse = process_sparse_feats(data, sparse_feats)

In [ ]:
total_data = pd.concat([data_dense, data_sparse], axis=1)
total_data['label'] = data['label']

# ???????

## ????dense?? 

In [ ]:
dense_inputs = []
for f in dense_feats:
    _input = Input([1],name=f)
    dense_inputs.append(_input)
    
    
concat_dense_inputs = Concatenate(axis=1)(dense_inputs)
concat_dense_inputs

## ????sparse??

In [ ]:
sparse_inputs = []
for f in sparse_feats:
    _input = Input([1],name=f)
    sparse_inputs.append(_input)

In [ ]:
# ???????
k = 8
sparse_kd_embed = []
for i, _input in enumerate(sparse_inputs):
    f = sparse_feats[i]
    voc_size = data[f].nunique()
    reg = tf.keras.regularizers.l2(0.7)
    _embed = Embedding(voc_size+1,k,embeddings_regularizer=reg)(_input)
    _embed = Flatten()(_embed)
    sparse_kd_embed.append(_embed)

In [ ]:
concat_sparse_inputs = Concatenate(axis=1)(sparse_kd_embed)
concat_sparse_inputs

## ??????sparse?dense ??

In [ ]:
embed_inputs = Concatenate(axis=1)([concat_sparse_inputs, concat_dense_inputs])
embed_inputs

## Cross Network

In [ ]:
def cross_layer(x0, x1):
    """
    ???? cross layer
    param x0: ??embeddings
    param x1: ????????
    
    """
    # 1. ??x1??embedding size
    embed_dim = x1.shape[-1]   # 221
    # 2. ???????W?b
    w = tf.Variable(tf.random.truncated_normal(shape=(embed_dim,), stddev=0.01))
    b = tf.Variable(tf.zeros(shape=(embed_dim,)))
    
    # 3.??feature crossing
    x_1w = tf.tensordot(tf.reshape(x1,[-1,1,embed_dim]),w,axes=1)
    cross = x0*x_1w
    return cross + b + x1

In [ ]:
def build_cross_layer(x0,num_layer=3):
    """
    ???? cross layer
    param x0: ?????embeddings
    param num_layers: cross net ??
    """
    # ??? x1?x0
    x1 = x0
    
    # ????cross net
    for i in range(num_layer):
        x1 = cross_layer(x0,x1)
        
    return x1    

In [ ]:
# cross net
cross_layer_output = build_cross_layer(embed_inputs, 1)

## DNN?? 

In [ ]:
fc_layer = Dropout(0.5)(Dense(128,activation='relu')(embed_inputs))
fc_layer = Dropout(0.3)(Dense(128,activation='relu')(fc_layer))
fc_layer_output = Dropout(0.1)(Dense(128,activation='relu')(fc_layer))

## ???? 

In [ ]:
cross_layer_output,fc_layer_output

In [ ]:
stack_layer = Concatenate()([cross_layer_output,fc_layer_output])

In [ ]:
output_layer = Dense(1,activation='sigmoid',use_bias=True)(stack_layer)

## ????

In [ ]:
model = Model(dense_inputs+sparse_inputs, output_layer)

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam',
             loss='binary_crossentropy',
             metrics=['binary_crossentropy',tf.keras.metrics.AUC(name='auc')])

In [ ]:

train_data = total_data.loc[:500000-1]
valid_data = total_data.loc[500000:]

train_dense_x = [train_data[f].values for f in dense_feats]
train_sparse_x = [train_data[f].values for f in sparse_feats]

train_label = [train_data['label'].values]


val_dense_x = [valid_data[f].values for f in dense_feats]
val_sparse_x = [valid_data[f].values for f in sparse_feats]


val_label = [valid_data['label'].values]

In [ ]:
model.fit(train_dense_x+train_sparse_x, train_label,
          epochs=5, batch_size=128,
         validation_data=(val_dense_x+val_sparse_x, val_label),
         )